# Bayesian Linear Regression with PyMC

## Learning Objectives

By completing this notebook, you will:
1. Build a complete Bayesian linear regression model in PyMC
2. Interpret posterior distributions for regression coefficients
3. Generate posterior predictive distributions for forecasting
4. Compare Bayesian results with frequentist OLS
5. Apply this to a commodity price modeling context

## Prerequisites
- Bayes' theorem and conjugate priors concepts
- Basic linear regression understanding
- PyMC environment verified

## Estimated Time: 90 minutes

---

In [ ]:
learning_objectives(["Build a complete Bayesian linear regression model in PyMC", "Interpret posterior distributions for regression coefficients", "Generate posterior predictive distributions for forecasting", "Compare Bayesian results with frequentist OLS", "Apply this to a commodity price modeling context", "Bayes' theorem and conjugate priors concepts"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

apply_course_theme()
apply_plot_theme()

## 1. Setup and Imports

First, let's import the libraries we'll use and set up our plotting style.

In [ ]:
# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Bayesian stack
import pymc as pm
import arviz as az

# For comparison
from scipy import stats
import statsmodels.api as sm

# Configuration
np.random.seed(42)
az.style.use('arviz-darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)

print(f"PyMC version: {pm.__version__}")
print(f"ArviZ version: {az.__version__}")

## 2. The Problem: Commodity Price Prediction

We'll model the relationship between crude oil prices and key fundamentals:
- **Inventory levels** (storage)
- **Production** (supply)

### Why Bayesian for this problem?

1. **Uncertainty quantification**: We need confidence intervals, not just point estimates
2. **Prior knowledge**: We have domain expertise about reasonable coefficient ranges
3. **Small samples**: Fundamental data often has limited history

Let's create some simulated data that mimics real commodity relationships.

In [ ]:
# Simulate commodity-like data
# True relationship: Price = 80 - 0.05*Inventory + 0.1*Production + noise

n_obs = 52  # One year of weekly data

# Fundamentals (standardized for easier interpretation)
inventory = np.random.normal(0, 1, n_obs)  # Standardized inventory deviation
production = np.random.normal(0, 1, n_obs)  # Standardized production

# True parameters
true_intercept = 80.0  # Base price level
true_beta_inventory = -5.0  # Higher inventory → lower price
true_beta_production = 3.0  # Higher production → higher price (counterintuitive but for demo)
true_sigma = 4.0  # Noise standard deviation

# Generate price
price = (
    true_intercept +
    true_beta_inventory * inventory +
    true_beta_production * production +
    np.random.normal(0, true_sigma, n_obs)
)

# Create DataFrame
data = pd.DataFrame({
    'price': price,
    'inventory': inventory,
    'production': production
})

print("Data Summary:")
print(data.describe())

In [ ]:
# Visualize the relationships
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Price vs Inventory
axes[0].scatter(data['inventory'], data['price'], alpha=0.6)
axes[0].set_xlabel('Inventory (standardized)')
axes[0].set_ylabel('Price')
axes[0].set_title('Price vs Inventory\n(Expected: negative relationship)')

# Price vs Production
axes[1].scatter(data['production'], data['price'], alpha=0.6)
axes[1].set_xlabel('Production (standardized)')
axes[1].set_ylabel('Price')
axes[1].set_title('Price vs Production')

plt.tight_layout()
plt.show()

## 3. Frequentist Baseline: OLS Regression

Before going Bayesian, let's see what ordinary least squares gives us.

In [ ]:
# OLS regression
X = sm.add_constant(data[['inventory', 'production']])
y = data['price']

ols_model = sm.OLS(y, X).fit()
print(ols_model.summary())

**Key observations from OLS:**
- Point estimates for coefficients
- Standard errors and p-values
- R-squared for model fit

**What's missing:**
- Full distribution of uncertainty
- Incorporation of prior knowledge
- Predictive distributions for new observations

## 4. Bayesian Linear Regression Model

Now let's build the Bayesian version. The model is:

$$y_i \sim \mathcal{N}(\mu_i, \sigma^2)$$
$$\mu_i = \alpha + \beta_1 x_{1i} + \beta_2 x_{2i}$$

With priors:
- $\alpha \sim \mathcal{N}(80, 20^2)$ — Price level around $80
- $\beta_j \sim \mathcal{N}(0, 10^2)$ — Coefficients could be positive or negative
- $\sigma \sim \text{HalfNormal}(10)$ — Noise scale (positive)

In [ ]:
# Build the Bayesian model
with pm.Model() as commodity_model:

    # --- Priors ---

    # Intercept: We expect prices around $80
    # Prior: N(80, 20²) gives 95% CI of roughly [40, 120]
    alpha = pm.Normal('alpha', mu=80, sigma=20)

    # Coefficient for inventory: Could be positive or negative
    # Prior: N(0, 10²) is weakly informative
    beta_inventory = pm.Normal('beta_inventory', mu=0, sigma=10)

    # Coefficient for production
    beta_production = pm.Normal('beta_production', mu=0, sigma=10)

    # Noise: Must be positive, HalfNormal is a good default
    sigma = pm.HalfNormal('sigma', sigma=10)

    # --- Likelihood ---

    # Expected value (linear combination)
    mu = alpha + beta_inventory * data['inventory'] + beta_production * data['production']

    # Observed data
    y_obs = pm.Normal('y_obs', mu=mu, sigma=sigma, observed=data['price'])

# Visualize model structure
pm.model_to_graphviz(commodity_model)

## 5. Prior Predictive Check

Before fitting, let's check if our priors produce sensible predictions.

**Why?** This catches unrealistic prior specifications before we waste computation.

In [ ]:
# Sample from the prior predictive distribution
with commodity_model:
    prior_predictive = pm.sample_prior_predictive(samples=1000, random_seed=42)

# Examine prior predictions
prior_preds = prior_predictive.prior_predictive['y_obs'].values.flatten()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(prior_preds, bins=50, density=True, alpha=0.7, label='Prior Predictive')
ax.axvline(data['price'].mean(), color='red', linestyle='--', linewidth=2,
           label=f'Data Mean: ${data["price"].mean():.1f}')
ax.set_xlabel('Price')
ax.set_ylabel('Density')
ax.set_title('Prior Predictive Check: Are our priors reasonable?')
ax.legend()

# Check coverage
pct_below_0 = (prior_preds < 0).mean() * 100
pct_above_200 = (prior_preds > 200).mean() * 100
print(f"Prior predictive: {pct_below_0:.1f}% below $0, {pct_above_200:.1f}% above $200")
print("(These should both be small for commodity prices)")

plt.show()

## 6. Fitting the Model: MCMC Sampling

Now we sample from the posterior distribution using MCMC (specifically, the NUTS sampler).

In [ ]:
# Sample from posterior
with commodity_model:
    # tune=1000: Initial samples to adapt sampler (discarded)
    # draws=2000: Samples we keep for inference
    # chains=4: Independent sampling chains for diagnostics
    trace = pm.sample(
        draws=2000,
        tune=1000,
        chains=4,
        cores=2,
        random_seed=42,
        return_inferencedata=True
    )

print("Sampling complete!")

## 7. Convergence Diagnostics

Before interpreting results, we must verify the sampler converged properly.

In [ ]:
# Summary statistics with diagnostics
summary = az.summary(trace, var_names=['alpha', 'beta_inventory', 'beta_production', 'sigma'])
print("Posterior Summary:")
print(summary)

**Key diagnostics to check:**

1. **r_hat** (R-hat): Should be < 1.01. Values > 1.01 suggest chains haven't converged.
2. **ess_bulk, ess_tail** (Effective Sample Size): Should be > 400. Low ESS means high autocorrelation.

If diagnostics look bad, increase `tune` and `draws`.

In [ ]:
# Trace plots: Visual check for mixing
az.plot_trace(trace, var_names=['alpha', 'beta_inventory', 'beta_production', 'sigma'])
plt.tight_layout()
plt.show()

**What good traces look like:**
- Left panels (posteriors): Smooth, unimodal distributions
- Right panels (chains): "Fuzzy caterpillars" - well-mixed, overlapping chains

## 8. Interpreting the Posterior

Now let's understand what the model learned.

In [ ]:
# Forest plot: Coefficients with credible intervals
az.plot_forest(
    trace,
    var_names=['beta_inventory', 'beta_production'],
    combined=True,
    hdi_prob=0.95
)
plt.axvline(0, color='red', linestyle='--', alpha=0.5)
plt.title('Coefficient Estimates (95% Credible Intervals)')
plt.show()

print("\nInterpretation:")
print(f"True beta_inventory: {true_beta_inventory}")
print(f"True beta_production: {true_beta_production}")

In [ ]:
# Posterior distributions
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

params = ['alpha', 'beta_inventory', 'beta_production', 'sigma']
true_values = [true_intercept, true_beta_inventory, true_beta_production, true_sigma]

for ax, param, true_val in zip(axes.flat, params, true_values):
    # Get posterior samples
    samples = trace.posterior[param].values.flatten()

    # Plot posterior
    ax.hist(samples, bins=50, density=True, alpha=0.7, label='Posterior')
    ax.axvline(true_val, color='red', linewidth=2, linestyle='--',
               label=f'True: {true_val}')
    ax.axvline(samples.mean(), color='green', linewidth=2,
               label=f'Post. Mean: {samples.mean():.2f}')
    ax.set_xlabel(param)
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('Posterior Distributions vs True Values', fontsize=14)
plt.tight_layout()
plt.show()

## 9. Comparison: Bayesian vs Frequentist

Let's directly compare OLS estimates with Bayesian posteriors.

In [ ]:
# Extract OLS results
ols_params = ols_model.params
ols_se = ols_model.bse

# Bayesian summaries
bayes_summary = az.summary(trace, var_names=['alpha', 'beta_inventory', 'beta_production'])

# Comparison table
comparison = pd.DataFrame({
    'True Value': [true_intercept, true_beta_inventory, true_beta_production],
    'OLS Estimate': ols_params.values,
    'OLS Std Error': ols_se.values,
    'Bayes Mean': bayes_summary['mean'].values,
    'Bayes Std': bayes_summary['sd'].values,
    'Bayes 95% HDI Low': bayes_summary['hdi_3%'].values,
    'Bayes 95% HDI High': bayes_summary['hdi_97%'].values
}, index=['alpha', 'beta_inventory', 'beta_production'])

print("Comparison: OLS vs Bayesian Regression")
print(comparison.round(3))

**Key differences:**

1. **Interpretation**: Bayesian gives P(parameter | data), frequentist gives P(data | parameter)
2. **Intervals**: Bayesian credible intervals have direct probability interpretation
3. **Regularization**: Priors provide natural regularization
4. **Predictions**: Full predictive distributions, not just point predictions

## 10. Posterior Predictive Checks

Does our model generate data that looks like the observed data?

In [ ]:
# Generate posterior predictive samples
with commodity_model:
    posterior_predictive = pm.sample_posterior_predictive(trace, random_seed=42)

# Plot posterior predictive check
az.plot_ppc(posterior_predictive, observed=True, num_pp_samples=100)
plt.title('Posterior Predictive Check')
plt.show()

**Interpretation:**
- Black line: Observed data distribution
- Blue lines: Simulated data from posterior predictive
- Good fit: Blue lines closely match the black line

## 11. Making Predictions

Now let's predict prices for new fundamental values.

In [ ]:
# New data: High inventory, low production scenario
new_inventory = 2.0  # 2 std above mean
new_production = -1.0  # 1 std below mean

# Get posterior samples
alpha_samples = trace.posterior['alpha'].values.flatten()
beta_inv_samples = trace.posterior['beta_inventory'].values.flatten()
beta_prod_samples = trace.posterior['beta_production'].values.flatten()
sigma_samples = trace.posterior['sigma'].values.flatten()

# Posterior predictive for new observation
# Mean prediction (without noise)
mu_pred = alpha_samples + beta_inv_samples * new_inventory + beta_prod_samples * new_production

# Full predictive (with noise)
y_pred = mu_pred + np.random.normal(0, sigma_samples)

# Summarize
print(f"Prediction for inventory={new_inventory}, production={new_production}:")
print(f"  Mean: ${np.mean(y_pred):.2f}")
print(f"  Std: ${np.std(y_pred):.2f}")
print(f"  95% Prediction Interval: [${np.percentile(y_pred, 2.5):.2f}, ${np.percentile(y_pred, 97.5):.2f}]")

In [ ]:
# Visualize the prediction
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(y_pred, bins=50, density=True, alpha=0.7, label='Predictive Distribution')
ax.axvline(np.mean(y_pred), color='red', linewidth=2, linestyle='--',
           label=f'Mean: ${np.mean(y_pred):.2f}')
ax.axvline(np.percentile(y_pred, 2.5), color='green', linewidth=2, linestyle=':',
           label='95% PI')
ax.axvline(np.percentile(y_pred, 97.5), color='green', linewidth=2, linestyle=':')

ax.set_xlabel('Predicted Price')
ax.set_ylabel('Density')
ax.set_title(f'Predictive Distribution\n(High Inventory, Low Production Scenario)')
ax.legend()
plt.show()

---

## Exercise 1: Change the Priors

**Task:** Re-fit the model with a stronger prior on `beta_inventory` that encodes the domain knowledge that higher inventory leads to lower prices.

Use: `beta_inventory ~ Normal(-5, 2)` instead of `Normal(0, 10)`.

**Questions:**
1. How does the posterior for `beta_inventory` change?
2. What happens to the credible interval width?

In [ ]:
# YOUR CODE HERE
# Build a new model with informative prior on beta_inventory

with pm.Model() as commodity_model_informative:
    # Priors
    alpha = pm.Normal('alpha', mu=80, sigma=20)

    # MODIFY THIS: Informative prior encoding domain knowledge
    beta_inventory = pm.Normal('beta_inventory', mu=0, sigma=10)  # <- Change this!

    beta_production = pm.Normal('beta_production', mu=0, sigma=10)
    sigma = pm.HalfNormal('sigma', sigma=10)

    # Likelihood
    mu = alpha + beta_inventory * data['inventory'] + beta_production * data['production']
    y_obs = pm.Normal('y_obs', mu=mu, sigma=sigma, observed=data['price'])

    # Sample
    trace_informative = pm.sample(2000, tune=1000, chains=2, cores=1,
                                   random_seed=42, progressbar=False)

# Compare posteriors
print("Weakly informative prior:")
print(az.summary(trace, var_names=['beta_inventory']))
print("\nInformative prior:")
print(az.summary(trace_informative, var_names=['beta_inventory']))

In [ ]:
# AUTO-GRADED TESTS
def test_exercise_1():
    """Verify informative prior was applied correctly."""
    # Check that the posterior mean for beta_inventory is negative
    post_mean = trace_informative.posterior['beta_inventory'].mean().values
    assert post_mean < 0, (
        f"beta_inventory posterior mean should be negative, got {post_mean:.2f}. "
        "Make sure you used an informative prior with negative mean."
    )

    # Check that uncertainty decreased (smaller std)
    weak_std = trace.posterior['beta_inventory'].std().values
    inform_std = trace_informative.posterior['beta_inventory'].std().values

    # Note: Informative prior might not always decrease std if data dominates
    print(f"Weak prior std: {weak_std:.3f}")
    print(f"Informative prior std: {inform_std:.3f}")

    print("\n✅ Exercise 1 completed!")

test_exercise_1()

---

## Exercise 2: Scenario Analysis

**Task:** Generate predictions for three scenarios and compare the predictive distributions:

1. **Base case:** inventory=0, production=0
2. **Supply glut:** inventory=2, production=2
3. **Supply crisis:** inventory=-2, production=-2

Create a plot showing all three predictive distributions.

In [ ]:
# YOUR CODE HERE
# Generate predictions for each scenario

scenarios = {
    'Base Case': (0, 0),
    'Supply Glut': (2, 2),
    'Supply Crisis': (-2, -2)
}

predictions = {}

for name, (inv, prod) in scenarios.items():
    # Calculate predictions using posterior samples
    # YOUR CODE: Calculate mu_pred and y_pred for each scenario
    mu_pred = None  # <- Fill this in
    y_pred = None   # <- Fill this in

    predictions[name] = y_pred

# Plot all three distributions
fig, ax = plt.subplots(figsize=(12, 6))

for name, preds in predictions.items():
    if preds is not None:
        ax.hist(preds, bins=50, density=True, alpha=0.5, label=name)

ax.set_xlabel('Predicted Price')
ax.set_ylabel('Density')
ax.set_title('Scenario Analysis: Predictive Distributions')
ax.legend()
plt.show()

---

## Summary

### Key Takeaways

1. **Bayesian regression** gives full posterior distributions over parameters
2. **Priors** encode domain knowledge and regularize estimates
3. **Convergence diagnostics** (R-hat, ESS, trace plots) must be checked
4. **Posterior predictive checks** verify model adequacy
5. **Predictive distributions** quantify forecast uncertainty

### What's Next

In the next notebook, we'll explore **prior sensitivity analysis** - how do our conclusions change with different prior choices?

### Additional Resources

- [PyMC Documentation](https://www.pymc.io/projects/docs/en/stable/)
- [ArviZ Documentation](https://arviz-devs.github.io/arviz/)
- McElreath, Chapter 4: "Geocentric Models"

---

*Remember: The posterior is not the truth - it's our best belief given the model and data. Always question your model!*